# MR DRE — Baseline trên Kaggle Free

**Mục tiêu:** tái lập quy trình benchmark MR DRE của bài báo ACL 2026 bằng tác nhân **LangChain Open Deep Research**, dùng **ResearcherBench (65 câu)**. Đây là baseline **trước** khi bổ sung ReviFence: không có kế hoạch chỉnh sửa cục bộ, không có tác nhân sửa riêng.

**Để tự chạy:** Tạo Kaggle Notebook → **File → Import Notebook** → chọn notebook này → trong **Settings**, chọn **GPU T4 x2** (nếu được cấp) và bật **Internet** → chọn **Save Version → Save & Run All**. Không cần GitHub, API key trả phí hay tải thêm ZIP. Có thể đóng trình duyệt sau khi Kaggle đã khởi chạy phiên nền.

**Mặc định** chạy 65 câu, phản hồi về nội dung `Content1`, hai lượt, chấm Coverage/Incorporation/Break Rate. Muốn thêm đánh giá trình bày và trích dẫn, thay `EXTRA_EVAL='none'` thành `'all'` ở ô cấu hình phía dưới; muốn phản hồi định dạng/tự suy xét, thay `FEEDBACK`. Mỗi tùy chọn thêm sẽ tốn thời gian.

Phiên GPU có giới hạn thời gian; notebook lưu checkpoint và tự dừng sớm. Nếu chưa đủ 65 câu, tải `baseline-results.zip` ở **Output**, tải lên Kaggle làm Dataset, **Add Input** vào notebook mới rồi bấm **Save Version → Save & Run All** lần nữa để tiếp tục, không tính lại câu đã làm. Không thể cam kết hoàn thành 65 câu trong một phiên miễn phí.

*Lưu ý nghiên cứu:* model Qwen3-4B và tìm kiếm miễn phí làm kết quả khác so với paper dùng GPT-4.1-series. Báo cáo phải ghi rõ đây là baseline tái lập **quy trình**, không phải số liệu tương đương paper.


In [ ]:
from pathlib import Path
import base64, io, zipfile, hashlib, time
SESSION_START = time.monotonic()
ROOT = Path('/kaggle/working/MRDRE-Baseline-Kaggle')
ROOT.parent.mkdir(parents=True, exist_ok=True)
PACKAGE = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE))) as z:
    assert z.testzip() is None
    z.extractall('/kaggle/working')
data = ROOT / 'data/ResearcherBench.jsonl'
assert hashlib.sha256(data.read_bytes()).hexdigest() == 'e345fe388af84ccb9cd1d92c52a7d409ab0ed1ca87f4279be8d9e8874e100391'
inputs = Path('/kaggle/input')
archives = sorted(inputs.rglob('baseline-results.zip')) if inputs.exists() else []
if archives:
    with zipfile.ZipFile(archives[-1]) as z:
        for item in z.namelist():
            assert not item.startswith('/') and '..' not in Path(item).parts, item
        z.extractall('/kaggle/working')
    print('Restored checkpoint:', archives[-1])
else:
    print('New baseline, 65 questions')
print('Source code:', ROOT)


## 1. Cài mô hình và khởi động dịch vụ chạy cục bộ


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, str(ROOT / 'kaggle/bootstrap.py')], cwd=ROOT, check=True)


## 2. Chạy 1 câu kiểm tra, sau đó chạy tối đa 65 câu


In [ ]:
import sys, subprocess, time
# Các tham số cố định dùng để so sánh baseline vs. ReviFence sau này.
N_QUESTIONS = 65  # giữ nguyên bộ ResearcherBench
FEEDBACK = 'content'  # content | format | reflect
EXTRA_EVAL = 'none'  # none: nhanh nhất | presentation | all: đầy đủ trích dẫn
SAFE_SESSION_MINUTES = 630  # tính từ thời điểm bắt đầu notebook, dừng trước 12 giờ

def execute(count, extra='none'):
    remaining = max(1, int(SAFE_SESSION_MINUTES - (time.monotonic() - SESSION_START) / 60))
    print(f'Start benchmark: {count} questions, feedback={FEEDBACK}, extras={extra}, time_budget={remaining}m')
    return subprocess.run([
        sys.executable, str(ROOT / 'kaggle/run.py'),
        '--count', str(count), '--feedback', FEEDBACK,
        '--chunk-size', '5' if count > 1 else '1',
        '--budget-minutes', str(remaining), '--extra-eval', extra,
    ], cwd=ROOT, check=False).returncode

# Nhất thiết phải hoàn thành một câu trước khi tiêu tốn quota cho toàn bộ dataset.
smoke = execute(1)
if smoke == 0:
    status = execute(N_QUESTIONS, EXTRA_EVAL)
    print('FULL RUN STATUS:', 'Completed' if status == 0 else 'Incomplete; checkpoint saved')
else:
    print('SMOKE TEST FAILED: run stopped; inspect baseline-logs and first failing stage.')


## 3. Tải kết quả và checkpoint để chạy tiếp


In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
work = Path('/kaggle/working')
print('Saved outputs:')
for f in sorted((work/'output').glob('*_scores.csv')):
    print(f.name, f.read_text()[:800])
for path in [work/'baseline-results.zip']:
    if path.exists():
        display(FileLink(str(path)))
    else:
        print('No checkpoint created yet.')
print('Detailed logs:', work/'baseline-logs')
